In [17]:
import os
os.chdir(r"S:\Projects for my portfolio\PowerPulse_Energy_Analysis")

In [18]:
import pandas as pd
gen = pd.read_csv(r"data\raw\organised_Gen.csv")
states = pd.read_csv(r"data\raw\states.csv")
print(gen.shape, states.shape)

(496774, 7) (51, 3)


In [19]:
gen = gen[gen["TYPE OF PRODUCER"] == "Total Electric Power Industry"]
gen = gen[gen["STATE"] != "US-TOTAL"]
gen = gen[~gen["ENERGY SOURCE"].isin(["Total", "Other", "Other Gases", "Pumped Storage"])]
gen["GENERATION (Megawatthours)"] = gen["GENERATION (Megawatthours)"].clip(lower=0)
print(gen.shape)

(95286, 7)


In [20]:
gen = gen.rename(columns={
    "YEAR": "year",
    "MONTH": "month",
    "STATE": "state_code",
    "ENERGY SOURCE": "energy_source",
    "GENERATION (Megawatthours)": "generation_mwh"
})
gen = gen[["year", "month", "state_code", "energy_source", "generation_mwh"]]
print(gen.columns.tolist())

['year', 'month', 'state_code', 'energy_source', 'generation_mwh']


In [21]:
import os
os.makedirs(r"data\processed", exist_ok=True)
print(os.listdir(r"data\processed"))

[]


In [22]:
gen.to_csv(r"data\processed\generation_clean.csv", index=False, encoding="utf-8")
print("generation_clean.csv saved!")

generation_clean.csv saved!


In [24]:
energy_sources = gen["energy_source"].unique()
print(energy_sources)

['Coal' 'Petroleum' 'Natural Gas' 'Hydroelectric Conventional' 'Wind'
 'Nuclear' 'Wood and Wood Derived Fuels' 'Other Biomass'
 'Solar Thermal and Photovoltaic' 'Geothermal']


In [25]:
fuel_map = {
    "Coal":                           ("Fossil",     0, 820.0),
    "Petroleum":                      ("Fossil",     0, 650.0),
    "Natural Gas":                    ("Fossil",     0, 490.0),
    "Nuclear":                        ("Low-Carbon", 0,  12.0),
    "Hydroelectric Conventional":     ("Renewable",  1,   4.0),
    "Wind":                           ("Renewable",  1,   0.0),
    "Solar Thermal and Photovoltaic": ("Renewable",  1,   0.0),
    "Geothermal":                     ("Renewable",  1,   0.0),
    "Wood and Wood Derived Fuels":    ("Renewable",  1,  50.0),
    "Other Biomass":                  ("Renewable",  1,  50.0),
}

fuel_types = pd.DataFrame([
    (source, *fuel_map[source]) for source in energy_sources
], columns=["energy_source", "category", "is_renewable", "carbon_intensity_gco2_per_kwh"])

print(fuel_types)

                    energy_source    category  is_renewable  \
0                            Coal      Fossil             0   
1                       Petroleum      Fossil             0   
2                     Natural Gas      Fossil             0   
3      Hydroelectric Conventional   Renewable             1   
4                            Wind   Renewable             1   
5                         Nuclear  Low-Carbon             0   
6     Wood and Wood Derived Fuels   Renewable             1   
7                   Other Biomass   Renewable             1   
8  Solar Thermal and Photovoltaic   Renewable             1   
9                      Geothermal   Renewable             1   

   carbon_intensity_gco2_per_kwh  
0                          820.0  
1                          650.0  
2                          490.0  
3                            4.0  
4                            0.0  
5                           12.0  
6                           50.0  
7                         

In [26]:
fuel_types.to_csv(r"data\processed\fuel_types.csv", index=False, encoding="utf-8")
print("fuel_types.csv saved!")

fuel_types.csv saved!


In [27]:
print(states.columns.tolist())
print(states.head())

['State', 'Abbrev', 'Code']
        State  Abbrev Code
0     Alabama    Ala.   AL
1      Alaska  Alaska   AK
2     Arizona   Ariz.   AZ
3    Arkansas    Ark.   AR
4  California  Calif.   CA


In [28]:
states.columns = ["state_name", "abbreviation", "state_code"]

region_map = {
    "CT":"Northeast","ME":"Northeast","MA":"Northeast","NH":"Northeast",
    "RI":"Northeast","VT":"Northeast","NJ":"Northeast","NY":"Northeast","PA":"Northeast",
    "IN":"Midwest","IL":"Midwest","MI":"Midwest","OH":"Midwest","WI":"Midwest",
    "IA":"Midwest","KS":"Midwest","MN":"Midwest","MO":"Midwest","NE":"Midwest","ND":"Midwest","SD":"Midwest",
    "DE":"South","FL":"South","GA":"South","MD":"South","NC":"South","SC":"South",
    "VA":"South","DC":"South","WV":"South","AL":"South","KY":"South","MS":"South",
    "TN":"South","AR":"South","LA":"South","OK":"South","TX":"South",
    "AZ":"West","CO":"West","ID":"West","MT":"West","NV":"West","NM":"West",
    "UT":"West","WY":"West","AK":"West","CA":"West","HI":"West","OR":"West","WA":"West",
}

states["region"] = states["state_code"].map(region_map).fillna("Other")
print(states.head(10))

             state_name abbreviation state_code     region
0               Alabama         Ala.         AL      South
1                Alaska       Alaska         AK       West
2               Arizona        Ariz.         AZ       West
3              Arkansas         Ark.         AR      South
4            California       Calif.         CA       West
5              Colorado        Colo.         CO       West
6           Connecticut        Conn.         CT  Northeast
7              Delaware         Del.         DE      South
8  District of Columbia         D.C.         DC      South
9               Florida         Fla.         FL      South


In [29]:
states.to_csv(r"data\processed\states_clean.csv", index=False, encoding="utf-8")
print("states_clean.csv saved!")

states_clean.csv saved!


In [30]:
sql_create = """
-- =====================================================
-- PowerPulse: US Energy Transition Analysis
-- 01_create_tables.sql
-- Creates 3 tables: states, fuel_types, generation
-- =====================================================

CREATE TABLE IF NOT EXISTS states (
    state_code    CHAR(2)      PRIMARY KEY,
    state_name    VARCHAR(50)  NOT NULL,
    abbreviation  VARCHAR(10),
    region        VARCHAR(20)
);

CREATE TABLE IF NOT EXISTS fuel_types (
    energy_source                 VARCHAR(50)  PRIMARY KEY,
    category                      VARCHAR(20),
    is_renewable                  SMALLINT,
    carbon_intensity_gco2_per_kwh NUMERIC(6,1)
);

CREATE TABLE IF NOT EXISTS generation (
    id             SERIAL        PRIMARY KEY,
    year           SMALLINT      NOT NULL,
    month          SMALLINT      NOT NULL,
    state_code     CHAR(2)       REFERENCES states(state_code),
    energy_source  VARCHAR(50)   REFERENCES fuel_types(energy_source),
    generation_mwh NUMERIC(15,2)
);
"""

with open(r"sql\01_create_tables.sql", "w", encoding="utf-8") as f:
    f.write(sql_create)

print("01_create_tables.sql saved!")

01_create_tables.sql saved!


In [36]:
sql_load = """
-- =====================================================
-- PowerPulse: US Energy Transition Analysis
-- 02_load_data.sql
-- =====================================================

-- Clear tables first to avoid duplicates
TRUNCATE TABLE generation, fuel_types, states CASCADE;

-- Load states first
COPY states(state_name, abbreviation, state_code, region)
FROM 'S:/Projects for my portfolio/PowerPulse_Energy_Analysis/data/processed/states_clean.csv'
CSV HEADER;

-- Load fuel_types second
COPY fuel_types(energy_source, category, is_renewable, carbon_intensity_gco2_per_kwh)
FROM 'S:/Projects for my portfolio/PowerPulse_Energy_Analysis/data/processed/fuel_types.csv'
CSV HEADER;

-- Load generation last
COPY generation(year, month, state_code, energy_source, generation_mwh)
FROM 'S:/Projects for my portfolio/PowerPulse_Energy_Analysis/data/processed/generation_clean.csv'
CSV HEADER;
"""

with open(r"sql\02_load_data.sql", "w", encoding="utf-8") as f:
    f.write(sql_load)

print("02_load_data.sql saved!")

02_load_data.sql saved!
